# 第31章 面积图（fill_between / stackplot）

使用填充区域表达连续趋势、区间范围或多个组成部分的累计变化。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

强调趋势的累计量、区间或随时间变化的组成。

## 数据结构

有序X轴和一条或多条非负序列；堆积面积图各序列单位相同。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 alpha 参数从 0.18 改为 0.5，观察透明度对填充区域可见性的影响
2. 修改 stackplot 中的 alpha 为 0.95，对比不透明堆积与透明堆积的视觉效果
3. 在 fill_between 中添加 where 参数（如 where=(sales > 150)），观察条件填充效果


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, color="#1a73e8", linewidth=2)
ax.fill_between(months, sales, color="#1a73e8", alpha=0.18)
ax.set(title="上半年销售额面积图", ylabel="销售额（万元）")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
office = np.array([38, 42, 40, 48, 55, 59])
digital = np.array([52, 65, 60, 78, 92, 105])
home = sales - office - digital
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.stackplot(months, office, digital, home, labels=["办公", "数码", "家居"], colors=["#8ab4f8", "#81c995", "#fdd663"], alpha=0.85)
ax.set(title="销售额品类构成变化", ylabel="销售额（万元）")
ax.legend(loc="upper left", frameon=False, ncol=3)
fig.tight_layout()
plt.show()


## 3. 参数说明

- alpha：透明度
- baseline：堆积基线
- labels：组成名称
- where：条件填充


## 4. 结果解读

普通面积图读取边界趋势；堆积面积图读取总高度和各层厚度。


## 常见误区

- 多层面积图难以比较中间序列
- 存在负值仍直接堆积
- 面积填充遮挡重要网格和文字


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
lower = sales * 0.9
upper = sales * 1.1
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, color="#188038", marker="o", label="预测")
ax.fill_between(months, lower, upper, color="#188038", alpha=0.18, label="±10%区间")
ax.set(title="销售预测及区间", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 本章小结

使用填充区域表达连续趋势、区间范围或多个组成部分的累计变化。


### 你已经掌握

- 判断面积图（fill_between / stackplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 强调趋势的累计量、区间或随时间变化的组成。 |
| 数据结构 | 有序X轴和一条或多条非负序列；堆积面积图各序列单位相同。 |
| 结果解读 | 普通面积图读取边界趋势；堆积面积图读取总高度和各层厚度。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `alpha` | 透明度 |
| `baseline` | 堆积基线 |
| `labels` | 组成名称 |
| `where` | 条件填充 |


### 需要注意

- 多层面积图难以比较中间序列
- 存在负值仍直接堆积
- 面积填充遮挡重要网格和文字


### 完成检查

- [ ] 能判断什么问题适合使用面积图（fill_between / stackplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
